<a href="https://colab.research.google.com/github/JWang-5779299/AnalyticsEngineering2026/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [2]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [3]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [4]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

## 1.1 What does one row of each table represent?


# Data Dictionary
| Column | Description |
|---|---|
| trip_id | Unique identifier for each trip record |
| bike_id | ID of the bike used for the trip |
| start_station_id | ID of the station where the trip started |
| end_station_id | ID of the station where the trip ended |
| start_time | Start timestamp of the trip |
| duration_min | Trip duration measured in minutes |
| user_type | Category of the user |
| bike_type | Type of bike: electric or standard |
| purchase_year | Year when the bike was purchased |
| station_id | Unique identifier for each station |
| station_name | Name of the station |
| zone | Zone that the station belongs to |
| capacity | Capacity of the station |


## 1.2 Which columns will you use?
We will use the following columns for the analysis:

- From `trips`:
  - `trip_id`: To count the total number of trips.
  - `bike_id`: Foreign key to join the `bikes` table.
  - `start_station_id`: Foreign key to join the `stations` table (we use the **start station zone** as requested).
  - `duration_min`: The target metric to calculate average trip duration in minutes.
- From `bikes`:
  - `bike_id`: Primary key to join with trips.
  - `bike_type`: Grouping variable (electric / standard bike).
- From `stations`:
  - `station_id`: Primary key to join with trips.
  - `zone`: Grouping variable for the start station’s zone.

## 1.3 What looks odd, missing or inconsistent? Write down what you find.

In [6]:
# 1. Load tables for inspection
trips = tables["trips"]
bikes = tables["bikes"]
stations = tables["stations"]

print("===== MISSING VALUES =====")
print("trips missing:\n", trips.isna().sum())
print("\nbikes missing:\n", bikes.isna().sum())
print("\nstations missing:\n", stations.isna().sum())

print("\n===== CHECK DUPLICATE PRIMARY KEYS =====")
print("trips duplicate trip_id:", trips["trip_id"].duplicated().sum())
print("bikes duplicate bike_id:", bikes["bike_id"].duplicated().sum())
print("stations duplicate station_id:", stations["station_id"].duplicated().sum())

print("\n===== DURATION SUMMARY (core metric) =====")
print(trips["duration_min"].describe())

print("\n===== UNIQUE BIKE TYPES =====")
print(bikes["bike_type"].value_counts())

print("\n===== UNIQUE ZONES =====")
print(stations["zone"].value_counts())


===== MISSING VALUES =====
trips missing:
 trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

bikes missing:
 bike_id          0
bike_type        0
purchase_year    0
dtype: int64

stations missing:
 station_id      0
station_name    0
zone            0
capacity        1
dtype: int64

===== CHECK DUPLICATE PRIMARY KEYS =====
trips duplicate trip_id: 12
bikes duplicate bike_id: 0
stations duplicate station_id: 0

===== DURATION SUMMARY (core metric) =====
count     582.000000
mean       27.891753
std       145.064305
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      2880.000000
Name: duration_min, dtype: float64

===== UNIQUE BIKE TYPES =====
bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64

===== UNIQUE ZONES =====
zone
Cent

### Findings
1. Missing values: 30 missing `duration_min` in trips; 1 missing `capacity` in stations (capacity unused, no impact).
2. Duplicates: 12 duplicated trip_id in trips (PK should be unique).
3. Duration: min=4 (reasonable), max=2880 mins (extreme outlier, unrealistic trip).
4. Bike_type: inconsistent labels: mixed upper/lower case, plus extra `E-bike` category.
5. Zone: zone labels are clean and consistent.

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

#### 2.1 Remove rows with missing `duration_min`. We cannot calculate average trip duration without this value, so these records are excluded.

In [7]:
trips_clean = trips.dropna(subset=["duration_min"])
print(f"After dropping missing duration: {len(trips_clean)} rows")


After dropping missing duration: 582 rows


#### 2.2 Remove duplicated `trip_id`. trip_id is primary key. Duplicates will overcount trips and distort results, keep only the first record.

In [8]:
trips_clean = trips_clean.drop_duplicates(subset=["trip_id"], keep="first")
print(f"After removing duplicate trip_id: {len(trips_clean)} rows")


After removing duplicate trip_id: 570 rows


#### 2.3 Clean bike_type labels. Convert all text to lowercase and map "e-bike" to "electric". This ensures only two groups: electric and standard.

In [9]:
bikes["bike_type_clean"] = bikes["bike_type"].str.lower()
bikes["bike_type_clean"] = bikes["bike_type_clean"].replace("e-bike", "electric")
print(bikes["bike_type_clean"].value_counts())


bike_type_clean
standard     47
electric     11
electric      2
Name: count, dtype: int64


#### 2.3.1 output two "electric", we should remove leading/trailing spaces, and map "e-bike" to "electric". Only two categories: electric and standard.


In [10]:
bikes["bike_type_clean"] = bikes["bike_type"].str.lower().str.strip()
bikes["bike_type_clean"] = bikes["bike_type_clean"].replace({"e-bike":"electric"})
print(bikes["bike_type_clean"].value_counts())


bike_type_clean
standard    47
electric    13
Name: count, dtype: int64


#### 2.4 Remove the 2880-minute outlier. A 48-hour bike trip is unrealistic and will pull up the average value.

In [11]:
trips_clean = trips_clean[trips_clean["duration_min"] < 2880]
print(trips_clean["duration_min"].describe())


count     569.000000
mean       23.042179
std        84.708126
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      1440.000000
Name: duration_min, dtype: float64


#### 2.5 Join cleaned trips, bikes and stations. Link trips to bikes via bike_id, link trips to stations via start_station_id

In [12]:
trips_bikes = trips_clean.merge(bikes, on="bike_id", how="left")
trips_final = trips_bikes.merge(stations, left_on="start_station_id", right_on="station_id", how="left")
print("Final cleaned dataset shape:", trips_final.shape)


Final cleaned dataset shape: (569, 14)


## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*